# 娇美人揣崽去逼婚，震惊家属院！ — v2

**Runtime → Run all**

In [ ]:
!pip install -q requests beautifulsoup4 chardet

In [ ]:
import re, requests, chardet
from bs4 import BeautifulSoup
from concurrent.futures import ThreadPoolExecutor, as_completed
from google.colab import files

MAIN_URL = 'http://www.pt9.net/txt209330.html'
OUTPUT   = '娇美人揣崽去逼婚_震惊家属院.txt'
WORKERS  = 8

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                  'AppleWebKit/537.36 (KHTML, like Gecko) '
                  'Chrome/124.0.0.0 Safari/537.36',
    'Accept-Language': 'zh-CN,zh;q=0.9',
    'Referer': 'http://www.pt9.net/',
}

# Patterns to skip (ads, navigation, watermarks)
SKIP = re.compile(
    r'(pt9\.net|请记住本书首发|www\.|本章未完|热门小说|推荐阅读|'
    r'上一章|下一章|最新章节|返回目录|加入书架|点击下一页|'
    r'笔趣阁|biquge|xbiquge|全本小说)',
    re.I
)

# Content selectors tried in order — more specific first
CONTENT_SELECTORS = [
    '#nr1', '#nr2', '#booktxt', '#htmlContent', '#BookText',
    '#readcontent', '#chaptercontent', '#content',
    '.read-content', '.content-body', '.chapter-content',
    'article', '.novel_content',
]

def smart_decode(resp):
    enc = chardet.detect(resp.content)['encoding'] or 'utf-8'
    try:
        return resp.content.decode(enc)
    except:
        return resp.content.decode('utf-8', errors='replace')

def count_chinese(text):
    return sum(1 for c in text if '\u4e00' <= c <= '\u9fff')

def extract_content(soup):
    """Try every selector; return element with the most Chinese chars."""
    best_el, best_count = None, 0
    for sel in CONTENT_SELECTORS:
        el = soup.select_one(sel)
        if el:
            cnt = count_chinese(el.get_text())
            if cnt > best_count:
                best_count = cnt
                best_el = el
    return best_el

def parse_content(el):
    if not el:
        return ''
    for tag in el.select('script,style,.ad,ins,a'):
        tag.decompose()
    lines = [p.get_text(' ', strip=True) for p in el.find_all(['p', 'br'])]
    if not lines:
        lines = [l.strip() for l in el.get_text('\n').splitlines() if l.strip()]
    # Keep only lines with meaningful Chinese content
    result = []
    for ln in lines:
        if SKIP.search(ln):
            continue
        if len(ln) < 4 and count_chinese(ln) == 0:
            continue
        result.append(ln)
    return '\n'.join(result)

# ── Step 1: get chapter list ──────────────────────────────────────────────────
print('Fetching chapter list…')
r = requests.get(MAIN_URL, headers=HEADERS, timeout=20)
html = smart_decode(r)
soup = BeautifulSoup(html, 'html.parser')

# Try common chapter-list selectors
links = []
for sel in ['.clist a', '.chapter-list a', '.mulu a', '#list a',
             'dl dd a', '.listmain a', '#chapterlist a',
             '#catalog a', '.catalog a', '.bd a']:
    links = soup.select(sel)
    if links:
        print(f'Chapter links found with selector: {sel}')
        break

if not links:
    # Fallback: any link matching chapter URL pattern
    links = [a for a in soup.find_all('a', href=True)
             if re.search(r'/(\d{3,}|chapter)[_/]?\d*\.html?$', a['href'])]
    print(f'Fallback: found {len(links)} pattern-matched links')

base = 'http://www.pt9.net'
chapters_urls = []
for a in links:
    href = a['href']
    if not href.startswith('http'):
        href = base + href
    title = a.get_text(strip=True)
    if title:
        chapters_urls.append((title, href))

print(f'Found {len(chapters_urls)} chapters')
if chapters_urls:
    print(f'  First: {chapters_urls[0]}')
    print(f'  Last:  {chapters_urls[-1]}')

if not chapters_urls:
    print('ERROR: No chapters found. Showing HTML sample for debugging:')
    print(html[:3000])

In [ ]:
# ── Step 2: diagnostic — check ONE chapter's HTML structure ───────────────────
if chapters_urls:
    test_url = chapters_urls[0][1]
    print(f'Checking first chapter: {test_url}')
    r2 = requests.get(test_url, headers=HEADERS, timeout=20)
    h2 = smart_decode(r2)
    s2 = BeautifulSoup(h2, 'html.parser')

    print('\n--- Content element scores (Chinese char count) ---')
    for sel in CONTENT_SELECTORS:
        el = s2.select_one(sel)
        if el:
            cnt = count_chinese(el.get_text())
            preview = el.get_text()[:80].replace('\n', ' ')
            print(f'  {sel:25s}  chars={cnt:4d}  preview: {preview}')

    el = extract_content(s2)
    content = parse_content(el)
    print(f'\n--- Best content (first 300 chars) ---')
    print(content[:300] if content else '(empty)')

In [ ]:
# ── Step 3: download all chapters ─────────────────────────────────────────────
def fetch_ch(args):
    i, title, url = args
    try:
        r = requests.get(url, headers=HEADERS, timeout=20)
        text = smart_decode(r)
        s = BeautifulSoup(text, 'html.parser')
        el = extract_content(s)
        content = parse_content(el)
        return i, title, content
    except Exception as e:
        return i, title, f'[error: {e}]'

results = {}
tasks = [(i+1, t, u) for i, (t, u) in enumerate(chapters_urls)]
with ThreadPoolExecutor(max_workers=WORKERS) as ex:
    futs = {ex.submit(fetch_ch, task): task for task in tasks}
    done = 0
    for f in as_completed(futs):
        i, title, content = f.result()
        results[i] = (title, content)
        done += 1
        if done % 50 == 0:
            print(f'  {done}/{len(tasks)} done')

with open(OUTPUT, 'w', encoding='utf-8') as f:
    for i, (title, content) in sorted(results.items()):
        f.write(f'\n\n第{i}章　{title}\n\n{content}\n')

print(f'\n✓ Done! Downloading file…')
files.download(OUTPUT)